# Anisotropic diffusion

This notebook accompanies Exercise 26.

Complete `EXERCISE_TENSOR_DIFFUSION` in `implementation_ueb4/myIntegrator.hpp`. The fixed-order $P_1$ element and space, quadrature setup, compilation, bindings and meshes are supplied.

Fill the two Python tasks below: compare the hand calculation with the local matrix, then compare the global matrices for the variable coefficient.

In [ ]:
from pathlib import Path
import sys
import numpy as np
from ngsolve import *
from ngsolve.fem import CompilePythonModule
from netgen.meshing import Mesh as NetgenMesh, MeshPoint, Pnt, Element2D, Element1D

lecture_dir = Path("../../notebooks/ch06_implementation").resolve()
sys.path.insert(0, str(lecture_dir))
from teaching_utils import square_mesh
m = CompilePythonModule(Path("implementation_ueb4/mymodule.cpp").resolve(),
                        init_function_name="teaching_fem")

## 26. Anisotropic diffusion

Implement the local matrix for

$$
\begin{align*}
\mathcal A_T(u,v)=\int_T (K\nabla u)\cdot\nabla v\,dx.
\end{align*}
$$

The reference derivative matrix stores the basis gradients as rows. Derive its transformation to physical gradients and the quadrature contribution to the local matrix. State the dimensions of all factors and identify where the area scaling enters.

First use the physical triangle with vertices $(2,0),(0,1),(0,0)$, corresponding to reference vertices $(1,0),(0,1),(0,0)$, and

$$
\begin{align*}
K=\begin{pmatrix}2&1\\1&3\end{pmatrix}.
\end{align*}
$$

Derive the gradients and matrix by hand before entering `tensor_expected`. Why must the constant vector belong to its kernel?

In [ ]:
triangle_mesh = NetgenMesh(dim=2)
domain = triangle_mesh.AddRegion("domain", dim=2)
boundary = triangle_mesh.AddRegion("boundary", dim=1)
triangle_vertices = [triangle_mesh.Add(MeshPoint(Pnt(a, b, 0)))
                     for a, b in [(2, 0), (0, 1), (0, 0)]]
triangle_mesh.Add(Element2D(domain, triangle_vertices))
for a, b in [(0, 1), (1, 2), (2, 0)]:
    triangle_mesh.Add(Element1D([triangle_vertices[a], triangle_vertices[b]], index=boundary))
tensor_mesh = Mesh(triangle_mesh)
tensor_fes = m.P1TriangleFESpace(tensor_mesh)
tensor_ei = ElementId(VOL, 0)
tensor_coefficient = CF((2, 1, 1, 3), dims=(2, 2))
tensor_matrix = np.array(m.TensorLaplace(tensor_coefficient).CalcElementMatrix(
    tensor_fes.GetFE(tensor_ei), tensor_mesh.GetTrafo(tensor_ei)))
print("C++ local matrix:\n", tensor_matrix)

In [ ]:
# TODO: enter the local matrix obtained by hand and compare it with tensor_matrix (np.linalg.norm).
tensor_expected = None

### A variable coefficient on a mesh

Compare your integrator with symbolic assembly on the supplied $P_1$ space, using

$$
\begin{align*}
K(x,y)=\begin{pmatrix}2+x&1\\1&3+y\end{pmatrix}.
\end{align*}
$$

Use quadrature order 12 for your integrator and `dx(bonus_intorder=10)` for the symbolic form. Measure the norm of the difference of the assembled matrices.

In [ ]:
mesh_tensor = square_mesh(3)
fes_tensor = m.P1TriangleFESpace(mesh_tensor)
variable_tensor = CF((2 + x, 1, 1, 3 + y), dims=(2, 2))
u, v = fes_tensor.TnT()
# TODO: assemble symbolic_tensor and own_tensor, then compare their matrices.
# Use .mat.ToDense() to convert the assembled matrices to dense format for comparison.